# 2. ESM-2: generate sequences from sequence context
**Question:** Can a model trained only on sequences suggest plausible new variants?

Start with ubiquitin, mask a few residues, and sample replacements with ESM-2. This is **generation around a seed sequence**, rather than generation from scratch. ESM-2 learns amino acid preferences from sequence context; we will inspect how its suggested changes affect the predicted fold.


## Setup
Select **Runtime → Change runtime type → T4 GPU**, then run the two cells below to load the tools and models.


In [ ]:
!git -C BME305-protein-design-tools pull -q 2>/dev/null || git clone -q https://github.com/RomeroLab/BME305-protein-design-tools.git
%pip -q install --upgrade ./BME305-protein-design-tools
from protein_design_tools import load_pdb, load_ESMFold, load_ESM2, load_MPNN, load_RFdiffusion, predict_structure, MPNNdesign, ESM2design, RFdiffusion, alignment, view_structure, inspect_prediction, save_results


In [ ]:
load_ESM2()
load_ESMFold();


## 1. Inspect the starting protein
We use the ubiquitin sequence as a seed. Its experimental structure is for inspection and comparison; ESM-2 receives only the sequence.


In [ ]:
native_seq, reference_pdb = load_pdb('1UBQ', chain='A')
view_structure(reference_pdb)


## 2. Generate variants
At each selected position, mask the residue and sample an amino acid from ESM-2's predictions. Each new sample becomes context for the next step. Higher temperature flattens the sampling distribution.


In [ ]:
designs = ESM2design(native_seq, num_sequences=3, n_mutations=5, temperature=1, seed=7)


## 3. Compare sequences
Which substitutions preserve chemical properties? Locate one substitution on the ubiquitin structure. Mutation labels use positions starting at 1.


In [ ]:
alignment({'ubiquitin':native_seq, **designs})


## 4. Predict and compare structures
Choose a design and predict its structure with ESMFold. The predictor uses the sequence alone, without the reference coordinates. RMSD measures agreement with the reference backbone; pLDDT describes confidence in the predicted structure. Consider both when interpreting the comparison.


In [ ]:
prediction_pdb = predict_structure(designs['design_1'], num_recycles=1)
metrics = inspect_prediction(reference_pdb, prediction_pdb)


## Discuss
- Does the predicted variant retain the overall fold? Are any changes near the sampled substitutions?
- How does sequence-only sampling differ from the backbone-conditioned ProteinMPNN demo?

[ESM-2 and ESMFold](https://github.com/facebookresearch/esm) · [1UBQ](https://www.rcsb.org/structure/1UBQ)


In [ ]:
bundle = save_results(designs, reference_pdb, prediction_pdb)
from google.colab import files
files.download(str(bundle))


The same tools accept your own sequences and structures. See the [function reference](https://github.com/RomeroLab/BME305-protein-design-tools#function-reference) for use in later assignments.
